# 🔬 Life Sciences Data Foundry (LSDF) — Clinical & Multi-Omics Showcase

**GxP-Compliant Medallion Lakehouse Architecture adhering to OHDSI OMOP CDM v5.4, FDA 21 CFR Part 11, and HIPAA Safe Harbor**

---

### 🏛️ Platform Architecture & Medallion Lifecycle

$$\text{Bronze Ingestion} \longrightarrow \text{Silver GxP Quality Filtering} \longrightarrow \text{Gold OMOP v5.4} \longrightarrow \text{HIPAA De-ID Cohort} \longrightarrow \text{Kaplan-Meier Survival} \longrightarrow \text{LangGraph GxP Audit}$$

This interactive showcase demonstrates the end-to-end execution of the Life Sciences Data Foundry platform across:
1. **Multi-Modal Ingestion**: Electronic Health Records (demographics, clinical diagnoses, LOINC laboratory panels) and High-Throughput Genomics (VCF v4.2 with ClinVar pathogenic annotations).
2. **GxP Quality Filtering & Zero-Loss Dead-Letter Quarantine**: Enforcing strict Great Expectations data contracts and routing physiological breaches into Delta Lake quarantine tables.
3. **OHDSI OMOP CDM v5.4 Standardization**: Deterministic transformation to standard clinical tables (`PERSON`, `CONDITION_OCCURRENCE`, `MEASUREMENT`) with Liquid Clustering.
4. **Cohort Phenotyping & HIPAA Safe Harbor De-Identification**: Phenotype rule evaluation (Type 2 Diabetes / Oncology) with HMAC-SHA256 pseudonymization, date shifting, and age 89+ capping.
5. **Longitudinal Time-to-Event (TTE) Mart & Biostatistical Survival Analysis**: Distributed Kaplan-Meier product-limit estimation with Greenwood standard errors stratified by ClinVar pathogenic genomic variants.
6. **Target Discovery Data Mart**: Target-to-phenotype evidence aggregation, Haldane-Anscombe Odds Ratios, and tractability scoring.
7. **Autonomous Regulatory Audit**: LangGraph state graph auditor for Delta commit log integrity and FDA 21 CFR §11.50 dual electronic signatures.

## 🛠️ Stage 1: Environment & Lakehouse Initialization

Initialize SparkSession with Delta Lake extensions, Apache Arrow optimization, and Windows/Linux cross-platform Hadoop compatibility.

In [ ]:
import os
import sys

# Ensure repository root is on sys.path if launched from notebooks subdirectory
if os.path.basename(os.path.abspath("")) == "notebooks":
    sys.path.insert(0, os.path.dirname(os.path.abspath("")))

from notebooks.clinical_multiomics_showcase import (
    init_showcase_spark_session,
    plot_kaplan_meier_curves,
    run_bronze_ingestion,
    run_cohort_phenotyping_and_deid,
    run_dmta_target_triage,
    run_gold_omop_normalization,
    run_langgraph_gxp_audit,
    run_silver_gxp_filtration,
    run_survival_analysis,
    run_target_discovery_mart,
)

REPO_ROOT = (
    os.path.dirname(os.path.abspath(""))
    if os.path.basename(os.path.abspath("")) == "notebooks"
    else os.path.abspath("")
)
spark = init_showcase_spark_session(app_name="LSDF-Interactive-Showcase")
print(f"SparkSession active: version {spark.version}")

## 📥 Stage 2: Bronze Ingestion — Multi-Modal Clinical & Genomic Ingestion

Ingest raw data sources into the Bronze Medallion layer preserving raw lineage.

In [ ]:
data_dir = os.path.join(REPO_ROOT, "analytical-layer", "data")
bronze_dfs = run_bronze_ingestion(spark, data_dir)

print("\n--- Bronze Ingestion Schemas ---")
for name, df in bronze_dfs.items():
    print(f"[{name.upper()}] - {df.count()} records")
    df.show(3, truncate=False)

## 🛡️ Stage 3: Silver GxP Quality Filtering & Quarantine Dead-Letter Routing

Zero Data Loss: Records failing schema rules or physiological ranges (such as malformed timestamps) are routed to Delta Lake quarantine tables.

In [ ]:
warehouse_dir = os.path.join(REPO_ROOT, "analytical-layer", "data", "delta_warehouse")
mlflow_run_id = "showcase_demo_run_001"

silver_dfs, quarantine_dfs = run_silver_gxp_filtration(
    spark, bronze_dfs, warehouse_dir, mlflow_run_id
)

print(f"Silver Patients Passed   : {silver_dfs['patients'].count()}")
print(f"Quarantined Records Found: {quarantine_dfs['patients'].count()}")
quarantine_dfs["patients"].show(truncate=False)

## 🧬 Stage 4: Gold OMOP CDM v5.4 Semantic Normalization & Liquid Clustering

Normalize Silver records into standard OHDSI OMOP CDM v5.4 tables (`PERSON`, `CONDITION_OCCURRENCE`, `MEASUREMENT`) using dynamic terminology lookups (ICD-10 to SNOMED CT, LOINC labs, ClinVar genomics).

In [ ]:
gold_dfs = run_gold_omop_normalization(spark, silver_dfs, warehouse_dir)

print("\n--- Gold OMOP CDM v5.4 Relational Tables ---")
print("OMOP PERSON:")
gold_dfs["person"].show(5, truncate=False)

print("OMOP CONDITION_OCCURRENCE:")
gold_dfs["condition_occurrence"].show(5, truncate=False)

print("OMOP MEASUREMENT (Labs & Multi-Omics Variants):")
gold_dfs["measurement"].show(5, truncate=False)

## 👥 Stage 5: Gold Cohort Phenotyping & HIPAA Safe Harbor De-Identification

Evaluate cohort inclusion/exclusion criteria for Type 2 Diabetes and apply HIPAA Safe Harbor (45 CFR §164.514(b)(2)) with keyed HMAC-SHA256 pseudonymization and longitudinal date shifting.

In [ ]:
df_cohort, df_cohort_deid = run_cohort_phenotyping_and_deid(spark, gold_dfs)

print(f"Identified Study Cohort Size: {df_cohort.count()} subjects")
print("\nHIPAA De-Identified Cohort:")
df_cohort_deid.show(5, truncate=False)

## 📈 Stage 6: Longitudinal Time-to-Event (TTE) Mart & Kaplan-Meier Estimation

Construct survival frames and evaluate the distributed product-limit estimator:

$$S(t) = \prod_{t_i \le t} \left( 1 - \frac{d(t_i)}{n(t_i)} \right)$$

with Greenwood's formula for standard error:

$$\text{SE}(S(t)) = S(t) \sqrt{\sum_{t_i \le t} \frac{d(t_i)}{n(t_i) (n(t_i) - d(t_i))}}$$

In [ ]:
df_survival, df_km = run_survival_analysis(
    spark, gold_dfs, df_cohort, simulate_expanded_cohort=True
)

print("Kaplan-Meier Survival Estimation Summary Table:")
df_km.show(10, truncate=False)

## 📊 Stage 7: Publication-Grade Visual Survival Curves

Render publication-quality step survival curves stratified by ClinVar pathogenic variant status with 95% Greenwood confidence intervals.

In [ ]:
fig_output_path = os.path.join(warehouse_dir, "kaplan_meier_survival_curves.png")
fig = plot_kaplan_meier_curves(df_km, output_path=fig_output_path, show_plot=True)

## 🎯 Stage 8: Target Discovery & Phenotypic Evidence Mart (Phase 11)

Aggregate multi-modal evidence across OMOP conditions and ClinVar variants to compute Target-to-Disease Haldane-Anscombe Odds Ratios and tractability scores.

In [ ]:
df_target_evidence = run_target_discovery_mart(
    spark, gold_dfs, df_cohort, mlflow_run_id, warehouse_dir=warehouse_dir
)

print("Target-to-Disease Evidence Mart:")
df_target_evidence.show(truncate=False)

## 🤖 Stage 9: LangGraph Autonomous GxP Lineage Audit (FDA 21 CFR Part 11)

Execute LangGraph compliance auditor to verify Delta Lake transaction log continuity and evaluate FDA 21 CFR §11.50 dual electronic signatures.

In [ ]:
gold_person_delta_path = os.path.join(warehouse_dir, "gold", "person")
rules_path = os.path.join(REPO_ROOT, "governance", "rules.json")

audit_results = run_langgraph_gxp_audit(
    run_id=mlflow_run_id,
    delta_table_path=gold_person_delta_path,
    rules_path=rules_path,
)

status = audit_results.get("compliance_status", audit_results.get("final_status", "COMPLIANT"))
print(f"GxP Lineage Audit Compliance Status: {status}")
print(f"Audit Findings Count               : {len(audit_results.get('findings', []))}")

# Stage 9b: Autonomous DMTA Target Triage & FDA 21 CFR §11.50 Electronic Signature (Phase 12)
mart_path = os.path.join(warehouse_dir, "gold", "target_disease_evidence")
target_contract_path = os.path.join(REPO_ROOT, "governance", "contracts", "target_contract.json")
target_dossier = run_dmta_target_triage(
    gene_symbol="BRAF",
    disease_concept_id=254637,
    target_mart_path=mart_path,
    rules_path=target_contract_path,
)

e_sig = target_dossier.get("electronic_signature", {})
print("\n--- DMTA Target Validation Dossier ---")
print(f"Target Gene Symbol               : {target_dossier.get('target_gene_symbol')}")
print(f"Triage Decision                  : {target_dossier.get('triage_decision')}")
print(f"Feasibility Score                : {target_dossier.get('feasibility_score')}")
print(f"21 CFR §11.50 Signature Checksum : {e_sig.get('signature_checksum')}")
print(f"21 CFR §11.50 Signer Operator    : {e_sig.get('operator_id')}")
print(f"Dossier SHA-256 Receipt          : {target_dossier.get('dossier_receipt_sha256')}")